In [1]:
from nd2reader import ND2Reader
import tifffile
import numpy as np
from matplotlib import pyplot as plt
import glob
import sys
import numpy as np
import os
import importlib

import skimage as sk
from skimage import filters, feature, exposure
from skimage.morphology import closing, disk
from skimage.filters import threshold_local, gaussian, threshold_multiotsu
from skimage.transform import rescale
from skimage.io import imread as tiff
from skimage import filters
import cv2


os.chdir('/Users/bebr1814/projects/anabaena/scratch_data/')

In [3]:
def otsu_thresholding(image):
    threshold_otsu = filters.threshold_otsu(image)
    mask_otsu = (image > threshold_otsu).astype(np.uint8) * 255
    return mask_otsu


def adaptive_thresholding(image):
    block_size = int(min(image.shape) / 20)
    
    # Ensure block_size is odd
    if block_size % 2 == 0:
        block_size += 1
    
    local_thresh = threshold_local(image, block_size, offset=10)
    mask_adaptive = (image > local_thresh).astype(np.uint8) * 255
    return mask_adaptive


def otsu_morphological_closing(image):
    mask_otsu = otsu_thresholding(image)
    selem = disk(5)
    mask_closing = closing(mask_otsu, selem)
    return mask_closing


def sobel_edge_detection(image):
    edges_sobel = filters.sobel(image)
    mask_sobel = (edges_sobel > 0.1).astype(np.uint8) * 255
    return mask_sobel


def canny_edge_detection(image):
    canny_edges = feature.canny(image, sigma=2)
    mask_canny = (canny_edges).astype(np.uint8) * 255
    return mask_canny


def gaussian_otsu(image):
    smoothed_image = gaussian(image, sigma=2)
    threshold_gaussian_otsu = filters.threshold_otsu(smoothed_image)
    mask_gaussian_otsu = (smoothed_image > threshold_gaussian_otsu).astype(
        np.uint8
    ) * 255
    return mask_gaussian_otsu


def histogram_equalization_otsu(image):
    equalized_image = exposure.equalize_hist(image)
    threshold_equalized_otsu = filters.threshold_otsu(equalized_image)
    mask_equalized_otsu = (equalized_image > threshold_equalized_otsu).astype(
        np.uint8
    ) * 255
    return mask_equalized_otsu


def clahe_otsu(image):
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    image_clahe = clahe.apply((image * 255).astype(np.uint8))
    threshold_clahe_otsu = filters.threshold_otsu(image_clahe)
    mask_clahe_otsu = (image_clahe > threshold_clahe_otsu).astype(np.uint8) * 255
    return mask_clahe_otsu


def filtered_contours_otsu(image):
    mask_otsu = otsu_thresholding(image)
    contours, _ = cv2.findContours(
        mask_otsu, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )
    min_contour_area = 100  # Adjust as needed
    filtered_contours = [c for c in contours if cv2.contourArea(c) > min_contour_area]
    mask_filtered_contours = np.zeros_like(mask_otsu)
    cv2.drawContours(
        mask_filtered_contours, filtered_contours, -1, 255, thickness=cv2.FILLED
    )
    return mask_filtered_contours


def multi_level_otsu(image):
    thresholds_multiotsu = threshold_multiotsu(
        image, classes=3
    )  # Adjust classes if needed
    mask_multiotsu = np.digitize(image, bins=thresholds_multiotsu) * 127
    return mask_multiotsu

In [26]:
def apply_mask_to_brightfield(fluorescent_image, brightfield_image,plot=False):
    # Ensure the mask and brightfield image are the same size
    assert (
        fluorescent_image.shape == brightfield_image.shape
    ), "The fluorescent and brightfield images must have the same dimensions."

    # Iterate through each of the masks and apply them to the brightfield image
    mask_functions = [
        # otsu_thresholding,
        # adaptive_thresholding,
        # otsu_morphological_closing,
        # sobel_edge_detection,
        # canny_edge_detection,
        gaussian_otsu,
        # histogram_equalization_otsu,
        # clahe_otsu,
        filtered_contours_otsu,
        # multi_level_otsu,
    ]

    for mask_function in mask_functions:
        # Generate the mask using the function
        mask = mask_function(fluorescent_image)

        # Ensure the mask is in the correct type and range for OpenCV
        mask = (mask > 0).astype(np.uint8)  # Convert mask to binary (0 or 1)

        # Apply the mask to the brightfield image
        masked_brightfield = cv2.bitwise_and(
            brightfield_image, brightfield_image, mask=mask
        )

        if plot:
            # Plot the fluorescent image, the mask, and the masked brightfield image
            plt.figure(figsize=(15, 5))

            # Fluorescent image
            plt.subplot(1, 3, 1)
            plt.imshow(fluorescent_image, cmap="gray")
            plt.title("Input Image")
            plt.axis("off")

            # Mask
            plt.subplot(1, 3, 2)
            plt.imshow(mask, cmap="gray")
            plt.title(f"Generated Mask -- {mask_function.__name__}")
            plt.axis("off")

            # Masked brightfield image
            plt.subplot(1, 3, 3)
            plt.imshow(masked_brightfield, cmap="gray")
            plt.title("Masked Brightfield Image (Background Removed)")
            plt.axis("off")

            plt.tight_layout()
            plt.show()
        
        return mask, masked_brightfield

In [41]:
folder = "/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB00*/tif/*"

# iterage through all the files in the folder and obtain the files that contain "*brightfield.tiff"
files_list = []
for filename in glob.glob(folder):
    if filename.endswith("brightfield.tiff"):
        files_list.append(filename)

print(files_list)

['/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/01.brightfield.tiff', '/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/19.brightfield.tiff', '/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/20.brightfield.tiff', '/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/14.brightfield.tiff', '/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/15.brightfield.tiff', '/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/09.brightfield.tiff', '/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/16.brightfield.tiff', '/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/12.br

In [38]:
def fill_in_euclidean(image, n):
    # Precompute the circular mask
    y, x = np.ogrid[-n:n+1, -n:n+1]
    mask = (x**2 + y**2 <= n**2)
    
    # Prepare the new image
    new_image = np.zeros_like(image)
    
    # Get the indices of all "1" pixels in the original image
    ones_indices = np.argwhere(image == 1)
    
    for i, j in ones_indices:
        # Define slice boundaries
        i_min = max(i - n, 0)
        i_max = min(i + n + 1, image.shape[0])
        j_min = max(j - n, 0)
        j_max = min(j + n + 1, image.shape[1])
        
        # Extract the region of interest and apply the mask
        new_image[i_min:i_max, j_min:j_max] = np.maximum(
            new_image[i_min:i_max, j_min:j_max],
            mask[(i_min - i + n):(i_max - i + n), (j_min - j + n):(j_max - j + n)]
        )
    
    return new_image

# # test
# image = np.array([[0, 0, 0, 0, 0, 0], [0, 0, 0, 1, 0, 0], [0, 0, 0, 0, 0, 0],[0, 0, 0, 0, 0, 0],[0, 0, 0, 0, 0, 0]])
# print(image)
# print(fill_in(image, 2))
# print(fill_in_euclidean(image, 2))

In [43]:
'/Users/bebr1814/projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/20241010_001_ZMB001/tif/01.brightfield.tiff'.split('/')[-1]

'01.brightfield.tiff'

In [55]:
# read the first file in the list
plot = True
save = False
save_path = '/scratch/Shares/anabaena_mcdb6440/fov_images/20241010_ZMB_Anabaena/'

for file in files_list:
    brightfield_image = tiff(file)
    cy5_img = tiff(file.replace("brightfield", "cy5"))
    texas_red_img = tiff(file.replace("brightfield", "texas_red"))

    # Apply the mask to the brightfield image
    mask, masked_brightfield = apply_mask_to_brightfield(cy5_img, brightfield_image, plot=False)

    mask_filled = fill_in_euclidean(mask, 50)
    
    masked_brightfield_filled = cv2.bitwise_and(brightfield_image, brightfield_image, mask=mask_filled)

    if plot:

        fig, axs = plt.subplots(1, 3, figsize=(15, 5))

        # Show original mask
        axs[0].imshow(mask, cmap="gray")
        axs[0].set_title("Original Mask")
        axs[0].axis("off")

        # Show filled mask
        axs[1].imshow(mask_filled, cmap="gray")
        axs[1].set_title("Filled Mask")
        axs[1].axis("off")

        # Show masked brightfield image
        axs[2].imshow(masked_brightfield_filled, cmap="gray")
        axs[2].set_title("Masked Brightfield Image")
        axs[2].axis("off")

        plt.tight_layout()
        savefile = os.path.join('/scratch/Shares/anabaena_mcdb6440/fov_images/plots',os.path.dirname(os.path.dirname(file))+'.'+os.path.basename(file).replace('.tiff','_masking.png'))
        plt.savefig(savefile,dpi=300, bbox_inches='tight')
        plt.close('all')

    if save:
        # write the masked_brightfield_filled
        save_file = os.path.join(save_path,file.split('/')[-3],'preprocessed',os.path.dirname(os.path.dirname(file)).split('_')[-1]+'.'+os.path.basename(file).replace('.tiff','_prep.tiff'))
        tifffile.imwrite(save_file, masked_brightfield_filled)

    # apply_mask_to_brightfield(texas_red_img, brightfield_image)
    # apply_mask_to_brightfield(brightfield_image, brightfield_image)

In [ ]:
# !cp projects/anabaena/scratch_data/fov_images/20241010_ZMB_Anabaena/*/preprocessed/* /Users/bebr1814/projects/anabaena/scratch_data/fov_images/training_data/

# !for f in *_prep.tiff; do if [ ! -f "${f/_prep/_labeled}" ]; then rm $f; fi;  done